# 01 — Validation

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

RAW_XLSX  = ROOT / "data" / "raw" / "dataset.xlsx"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
assert RAW_XLSX.exists(), f"Missing dataset at {RAW_XLSX}"


## 1. Load the four canonical sheets

In [ ]:
card           = pd.read_excel(RAW_XLSX, sheet_name="card")
transaction    = pd.read_excel(RAW_XLSX, sheet_name="transaction")
cost_structure = pd.read_excel(RAW_XLSX, sheet_name="cost_structure")
rates          = pd.read_excel(RAW_XLSX, sheet_name="rates")

# Drop trailing all-NaN columns left over from xlsx formatting
transaction = transaction.dropna(axis=1, how="all")

for name, df in [("card", card), ("transaction", transaction),
                 ("cost_structure", cost_structure), ("rates", rates)]:
    print(f"{name:18s} rows={len(df):>6,}   cols={df.shape[1]}")

## 2. Schema dump

In [ ]:
for name, df in [("card", card), ("transaction", transaction),
                 ("cost_structure", cost_structure), ("rates", rates)]:
    print(f"\n=== {name} ===")
    df.info()

## 3. Type normalisation

In [ ]:
for df in (card, transaction, cost_structure, rates):
    for col in df.select_dtypes(include=["object", "str"]).columns:
        df[col] = df[col].astype("string")
print("Normalised object columns to pandas `string` dtype.")

## 5. PK uniqueness and referential integrity

In [ ]:
# PK uniqueness
checks = [
    ("card.card_token",       card["card_token"].is_unique),
    ("transaction.id",        transaction["id"].is_unique),
    ("rates.code",            rates["code"].is_unique),
    ("cost_structure (cost_type, region, cost_line)",
        not cost_structure.duplicated(subset=["transaction_cost_type", "cost_region", "cost_line"]).any()),
]
for name, ok in checks:
    print(f"  {'✓' if ok else '✗'} {name}")

In [ ]:
# Foreign-key integrity
orphan_tx = (~transaction["card_token"].isin(card["card_token"])).sum()
print(f"  transaction.card_token → card: {orphan_tx} orphans")

tx_pairs   = transaction[["transaction_cost_type", "cost_region"]].drop_duplicates()
cost_pairs = cost_structure[["transaction_cost_type", "cost_region"]].drop_duplicates()
unmatched = tx_pairs.merge(cost_pairs, how="left", indicator=True).query("_merge == 'left_only'")
print(f"  transaction (cost_type, region) → cost_structure: {len(unmatched)} unmatched pairs")

referenced_ccys = pd.unique(pd.concat([
    transaction["amount_currency"],
    transaction["billing_amount_currency"],
    transaction["interchange_currency"],
]).dropna())
missing_ccys = set(referenced_ccys) - set(rates["code"])
print(f"  transaction currencies → rates.code: {len(missing_ccys)} missing ({missing_ccys or 'none'})")

## 6. Missing-values audit

In [ ]:
def null_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    n = len(df)
    rows = [
        {"table": name, "column": col,
         "null_count": int(df[col].isna().sum()),
         "pct": round(100 * df[col].isna().sum() / n, 2)}
        for col in df.columns if df[col].isna().any()
    ]
    return pd.DataFrame(rows)

pd.concat([
    null_report(card, "card"),
    null_report(transaction, "transaction"),
    null_report(cost_structure, "cost_structure"),
    null_report(rates, "rates"),
], ignore_index=True)

## 7. Persist as CSV

In [ ]:
card.to_csv(PROCESSED / "card.csv", index=False)
transaction.to_csv(PROCESSED / "transaction.csv", index=False)
cost_structure.to_csv(PROCESSED / "cost_structure.csv", index=False)
rates.to_csv(PROCESSED / "rates.csv", index=False)

for p in sorted(PROCESSED.glob("*.csv")):
    print(f"  {p.name:25s} {p.stat().st_size / 1024:>7.1f} KB")